# ACT 50-demo baseline on Kaggle

Prompt2MotionのPBS/MI210用処理を使わず、KaggleのNVIDIA GPUでACTの100 epoch smoke testを実行するノートブックです。Kaggle NotebookのSettingsでGPUとInternetを有効にし、50 demosのZIPまたは展開済みHDF5をInputへ追加してから、上から順に実行してください。

データセット、checkpoint、動画はGitへ追加しません。Notebook終了前に最後のセルで成果物ZIPを作成し、Outputとして保存してください。

In [ ]:
# 1. NVIDIA GPU / CUDAの確認
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=False)
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "Kaggle NotebookのSettingsでGPU acceleratorを有効にしてください"

## 2. Prompt2MotionとACTの取得

ACTは検証済みcommitへ固定し、Prompt2Motionに保存したローカル改修パッチを適用します。セルは再実行可能です。

In [ ]:
from pathlib import Path
import os
import subprocess

WORKING = Path("/kaggle/working")
PROJECT = WORKING / "Prompt2motion"
ACT_REPO = PROJECT / "repos" / "act"
ACT_COMMIT = "742c753c0d4a5d87076c8f69e5628c79a8cc5488"
ACT_PATCH = PROJECT / "patches" / "act-local.patch"

if not (PROJECT / ".git").is_dir():
    subprocess.run([
        "git", "clone",
        "https://github.com/Hatano123/Prompt2motion.git",
        str(PROJECT),
    ], check=True)

ACT_REPO.parent.mkdir(parents=True, exist_ok=True)
if not (ACT_REPO / ".git").is_dir():
    subprocess.run([
        "git", "clone",
        "https://github.com/tonyzhaozh/act.git",
        str(ACT_REPO),
    ], check=True)
    subprocess.run(["git", "-C", str(ACT_REPO), "checkout", ACT_COMMIT], check=True)
else:
    current_commit = subprocess.check_output(
        ["git", "-C", str(ACT_REPO), "rev-parse", "HEAD"], text=True
    ).strip()
    assert current_commit == ACT_COMMIT, f"ACT commit mismatch: {current_commit}"

assert ACT_PATCH.is_file(), f"必須パッチがありません: {ACT_PATCH}"
already_applied = subprocess.run(
    ["git", "-C", str(ACT_REPO), "apply", "--reverse", "--check", str(ACT_PATCH)],
    capture_output=True,
).returncode == 0
if not already_applied:
    subprocess.run(["git", "-C", str(ACT_REPO), "apply", "--check", str(ACT_PATCH)], check=True)
    subprocess.run(["git", "-C", str(ACT_REPO), "apply", str(ACT_PATCH)], check=True)

print("Prompt2Motion:", PROJECT)
print("ACT:", ACT_REPO)
print("ACT commit:", ACT_COMMIT)
print("ACT patch:", "already applied" if already_applied else "applied now")
subprocess.run(["git", "-C", str(ACT_REPO), "status", "--short"], check=True)

## 3. Python依存関係

Kaggle組込みのCUDA版PyTorchは置き換えません。Prompt2MotionのrequirementsとACT/DETRだけを追加します。pip実行後にdependency warningが出た場合は内容を確認してください。

In [ ]:
# 既存のCUDA版torchを維持するため、ROCm wheelはインストールしない
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(PROJECT / "requirements-act.txt"),
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    "-e", str(ACT_REPO / "detr"),
], check=True)

import mujoco
import dm_control
import h5py
import einops

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("mujoco:", mujoco.__version__)
print("Imports OK")

## 4. 50 demosデータの検出と配置

KaggleのAdd Inputから次のどちらかを追加してください。

- `act_sim_transfer_cube_scripted_50episodes.zip`
- `episode_0.hdf5` ～ `episode_49.hdf5` を含む展開済みディレクトリ

ZIPは `/kaggle/working/act_data` へ展開します。展開済みInputはread-onlyのままsymlinkで参照します。

In [ ]:
import shutil
import zipfile

KAGGLE_INPUT = Path("/kaggle/input")
DATA_ROOT = WORKING / "act_data"
EXPECTED_DIR = DATA_ROOT / "sim_transfer_cube_scripted"
EXPECTED_ZIP = "act_sim_transfer_cube_scripted_50episodes.zip"

def valid_dataset(directory: Path) -> bool:
    return directory.is_dir() and all(
        (directory / f"episode_{index}.hdf5").is_file()
        for index in range(50)
    )

DATA_ROOT.mkdir(parents=True, exist_ok=True)
if not valid_dataset(EXPECTED_DIR):
    zip_candidates = list(KAGGLE_INPUT.rglob(EXPECTED_ZIP))
    if zip_candidates:
        print("Extracting:", zip_candidates[0])
        with zipfile.ZipFile(zip_candidates[0]) as archive:
            archive.extractall(DATA_ROOT)
    else:
        episode_zero_candidates = list(KAGGLE_INPUT.rglob("episode_0.hdf5"))
        source_dir = next(
            (path.parent for path in episode_zero_candidates if valid_dataset(path.parent)),
            None,
        )
        if source_dir is None:
            sample_files = [str(path) for path in list(KAGGLE_INPUT.rglob("*"))[:50] if path.is_file()]
            raise FileNotFoundError(
                "50 demosのZIPまたはepisode_0～49.hdf5が見つかりません。"
                f" Add Inputを確認してください。先頭のInput files: {sample_files}"
            )
        if EXPECTED_DIR.exists() or EXPECTED_DIR.is_symlink():
            raise RuntimeError(f"不完全な既存データを退避して再実行してください: {EXPECTED_DIR}")
        EXPECTED_DIR.symlink_to(source_dir, target_is_directory=True)

assert valid_dataset(EXPECTED_DIR), f"episode_0～49を確認できません: {EXPECTED_DIR}"
episode_files = sorted(EXPECTED_DIR.glob("episode_*.hdf5"))
print("Dataset directory:", EXPECTED_DIR)
print("HDF5 files found:", len(episode_files))
print("Total GiB:", round(sum(path.stat().st_size for path in episode_files) / 1024**3, 2))

os.environ["ACT_DATA_DIR"] = str(DATA_ROOT)
os.environ["ACT_NUM_EPISODES"] = "50"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

## 5. ACT 100 epoch smoke training

元のMI210 baselineと同じ主要hyperparameterを使います。同じセルを再実行した場合、`training_state_last.ckpt` があれば自動resumeします。100 epochは性能確認ではなく、配線と学習完走の確認です。

In [ ]:
EXPERIMENT = "act50_kaggle_100e"
NUM_EPOCHS = 100
BATCH_SIZE = 8
SAVE_EVERY = 20
CKPT_DIR = WORKING / "checkpoints" / EXPERIMENT
CKPT_DIR.mkdir(parents=True, exist_ok=True)

train_command = [
    sys.executable, str(ACT_REPO / "imitate_episodes.py"),
    "--task_name", "sim_transfer_cube_scripted",
    "--ckpt_dir", str(CKPT_DIR),
    "--policy_class", "ACT",
    "--kl_weight", "10",
    "--chunk_size", "100",
    "--hidden_dim", "512",
    "--batch_size", str(BATCH_SIZE),
    "--dim_feedforward", "3200",
    "--num_epochs", str(NUM_EPOCHS),
    "--max_epochs_this_run", str(NUM_EPOCHS),
    "--save_every", str(SAVE_EVERY),
    "--lr", "1e-5",
    "--seed", "0",
]
if (CKPT_DIR / "training_state_last.ckpt").is_file():
    train_command.append("--resume")
    print("Resuming from:", CKPT_DIR / "training_state_last.ckpt")

print("Running:", " ".join(train_command))
subprocess.run(train_command, cwd=ACT_REPO, env=os.environ.copy(), check=True)

## 6. Best checkpointのrollout評価

既定は10 rolloutsです。時間が厳しい場合は `EVAL_ROLLOUTS = 1` に変更してください。

In [ ]:
EVAL_ROLLOUTS = 10
EVAL_DIR = WORKING / "results" / EXPERIMENT / "eval_best"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
assert (CKPT_DIR / "policy_best.ckpt").is_file(), "学習済みpolicy_best.ckptがありません"

eval_command = [
    sys.executable, str(ACT_REPO / "imitate_episodes.py"),
    "--eval",
    "--eval_ckpt", "policy_best.ckpt",
    "--eval_output_dir", str(EVAL_DIR),
    "--num_rollouts", str(EVAL_ROLLOUTS),
    "--task_name", "sim_transfer_cube_scripted",
    "--ckpt_dir", str(CKPT_DIR),
    "--policy_class", "ACT",
    "--kl_weight", "10",
    "--chunk_size", "100",
    "--hidden_dim", "512",
    "--batch_size", str(BATCH_SIZE),
    "--dim_feedforward", "3200",
    "--num_epochs", str(NUM_EPOCHS),
    "--lr", "1e-5",
    "--seed", "0",
]
print("Running:", " ".join(eval_command))
subprocess.run(eval_command, cwd=ACT_REPO, env=os.environ.copy(), check=True)
print((EVAL_DIR / "result_policy_best.txt").read_text())

## 7. 成果物をKaggle Output用ZIPへまとめる

NotebookのSave Version時にOutputへ保存するか、生成されたZIPを手元へダウンロードしてください。

In [ ]:
import json

manifest = {
    "experiment": EXPERIMENT,
    "act_commit": ACT_COMMIT,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "eval_rollouts": EVAL_ROLLOUTS,
}
(WORKING / f"{EXPERIMENT}_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

bundle_root = WORKING / "kaggle_act_artifacts"
bundle_root.mkdir(exist_ok=True)
shutil.copytree(CKPT_DIR, bundle_root / "checkpoints", dirs_exist_ok=True)
shutil.copytree(EVAL_DIR, bundle_root / "evaluation", dirs_exist_ok=True)
shutil.copy2(WORKING / f"{EXPERIMENT}_manifest.json", bundle_root / "manifest.json")
archive_path = shutil.make_archive(
    str(WORKING / f"{EXPERIMENT}_artifacts"), "zip", root_dir=bundle_root
)
print("Artifact archive:", archive_path)
print("Size MiB:", round(Path(archive_path).stat().st_size / 1024**2, 1))

## 2000 epochへ進む場合

100 epochの完走、loss、GPU memory、評価出力を確認してから `EXPERIMENT` を新しい名前、`NUM_EPOCHS = 2000`、`SAVE_EVERY = 100` に変更してください。Kaggleのsession制限をまたぐ場合は、最後の成果物ZIPから `training_state_last.ckpt` を含むcheckpoint一式を次sessionの同じ `CKPT_DIR` へ復元してから学習セルを再実行します。